[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anicka-net/nla-at-home/blob/main/notebooks/02_injection_mechanism.ipynb)

# 02 · The Injection Mechanism

### HAAISS workshop — core notebook 2 of 4

In notebook 01 `describe()` was a black box. Here you **build it** and then break several parts of its interface deliberately.

The whole trick: a language model consumes a sequence of **embedding vectors**. We put a throwaway placeholder token in the prompt, then overwrite *its* embedding with the activation we want the model to describe. The model can't tell the difference between a real token embedding and our smuggled-in vector — as long as the vector looks like the ones it was trained on.

## Setup (same as notebook 01)

In [1]:
!pip install -q -U transformers peft accelerate bitsandbytes


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE       = "Qwen/Qwen2.5-7B-Instruct"          # the model whose mind we read
AV_ADAPTER = "anicka/nla-qwen2.5-7b-universal-av-grpo"  # universal verbalizer: ONE adapter, all 28 layers (GRPO-refined)
LAYER      = 20   # default readout layer — the universal adapter serves ALL of 0..27; try others
DEPTH_PCT  = 71   # nearest trained depth tag for layer 20; a conditioning input
INJECT_CHAR  = "\u320e"                          # the placeholder token we overwrite: ㈎
INJECT_SCALE = 150.0                              # we normalize the activation's L2 norm TO this

/home/anicka/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = "cuda"
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"

# 4-bit so a 7B model + adapters fit a free-Colab T4 (16 GB). fp16 compute:
# the GRPO-sharpened adapter is numerically sensitive, and fp16 on CUDA is a
# tested-safe path (bf16 on Apple MPS collapses it; not our case here).
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)

tok  = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb,
                                            device_map={"": 0})
model = PeftModel.from_pretrained(base, AV_ADAPTER).eval()   # adapter name = "default"

inject_id = tok.encode(INJECT_CHAR, add_special_tokens=False)
assert len(inject_id) == 1, f"injection char must be ONE token, got {inject_id}"
inject_id = inject_id[0]
print("loaded — base + AV adapter on", next(model.parameters()).device)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/339 [00:06<38:11,  6.78s/it]

Loading weights:   1%|          | 2/339 [00:13<38:05,  6.78s/it]

Loading weights:   1%|          | 4/339 [00:14<15:40,  2.81s/it]

Loading weights:   1%|▏         | 5/339 [00:15<12:15,  2.20s/it]

...

Loading weights:  97%|█████████▋| 330/339 [01:28<00:03,  2.76it/s]

Loading weights:  99%|█████████▊| 334/339 [01:28<00:01,  4.65it/s]

Loading weights:  99%|█████████▉| 336/339 [01:28<00:00,  5.51it/s]

Loading weights: 100%|██████████| 339/339 [01:28<00:00,  3.81it/s]

loaded — base + AV adapter on cuda:0


In [4]:
def get_layers(m):
    """Reach the transformer block list through the PEFT + CausalLM wrappers."""
    b = m.base_model.model if hasattr(m, "base_model") else m
    inner = b.model if hasattr(b, "model") else b
    return inner.layers

def read_activation(prompt, layer=LAYER, max_new_tokens=128):
    """Grab the clean base-model residual at the last prompt token."""
    chat = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                   tokenize=False, add_generation_prompt=True)
    inp = tok(chat, return_tensors="pt").to(device)

    grab = {}
    def hook(mod, inpt, out):
        h = out[0] if isinstance(out, tuple) else out
        if "h" not in grab:                 # FIRST forward pass only — otherwise
            grab["h"] = h[:, -1, :].detach() # every generated token overwrites it
    handle = get_layers(model)[layer].register_forward_hook(hook)
    try:
        with model.disable_adapter(), torch.no_grad():
            out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False,
                                 pad_token_id=tok.eos_token_id)
    finally:
        handle.remove()
    reply = tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)
    return grab["h"].squeeze(0), reply

def normalize_to(v, scale=INJECT_SCALE):
    """Rescale v so its L2 norm equals `scale`. NOT v * scale — see notebook 02."""
    n = v.float().norm().clamp_min(1e-12)
    return v * (scale / n)

def av_prompt(depth_pct):
    return (
        "You are a meticulous AI researcher conducting an important investigation "
        "into activation vectors from a language model. Your overall task is to "
        "describe the semantic content of that activation vector.\n\n"
        "We will pass the vector enclosed in <concept> tags into your context, "
        "along with the network depth where it was extracted. "
        "You must then produce an explanation for the vector, enclosed within "
        "<explanation> tags. The explanation consists of 2-3 text snippets "
        "describing that vector.\n\n"
        f"Here is the vector from depth {depth_pct}% of the network:\n\n"
        f"<concept>{INJECT_CHAR}</concept>\n\n"
        "Please provide an explanation.\n\n"
        "<explanation>")

def describe(activation, depth=DEPTH_PCT, max_new_tokens=120, scale_fn=normalize_to):
    """The whole NLA read: build the prompt, overwrite the placeholder token's
    embedding with the (rescaled) activation, let the model narrate."""
    chat = tok.apply_chat_template([{"role": "user", "content": av_prompt(depth)}],
                                   tokenize=False, add_generation_prompt=True)
    ids = tok.encode(chat, add_special_tokens=False)  # match training: chat-wrapped, no BOS
    pos = ids.index(inject_id)
    input_ids = torch.tensor([ids], device=device)
    emb = model.get_input_embeddings()(input_ids).clone()
    emb[0, pos, :] = scale_fn(activation.to(emb.dtype))
    attn = torch.ones((1, len(ids)), device=device, dtype=torch.long)
    with torch.no_grad():
        out = model.generate(input_ids=input_ids, inputs_embeds=emb, attention_mask=attn,
                             max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tok.eos_token_id)
    seq = out[0]
    gen = seq[len(ids):] if seq.shape[0] > len(ids) else seq  # embeds path returns new-only
    return tok.decode(gen, skip_special_tokens=True).split("</explanation>")[0].strip()

## Build the injection by hand

Forget `describe()` for a moment. Here is the mechanism, unrolled, with the activation of a real prompt.

In [5]:
activation, _ = read_activation("Explain how a hash map handles collisions.")
print("raw activation L2 norm:", round(activation.float().norm().item(), 1))

# 1) build the prompt text with the placeholder char in it
chat = tok.apply_chat_template([{"role": "user", "content": av_prompt(DEPTH_PCT)}],
                               tokenize=False, add_generation_prompt=True)
ids = tok.encode(chat, add_special_tokens=False)  # match training: chat-wrapped, no BOS
pos = ids.index(inject_id)          # where the placeholder landed
print("placeholder token id:", inject_id, "at position", pos)

# 2) turn tokens into embeddings
input_ids = torch.tensor([ids], device=device)
emb = model.get_input_embeddings()(input_ids).clone()
print("one embedding row looks like:", tuple(emb[0, pos].shape),
      "norm", round(emb[0, pos].float().norm().item(), 1))

/home/anicka/venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


raw activation L2 norm: 120.0
placeholder token id: 149705 at position 130
one embedding row looks like: (3584,) norm 0.1


### The critical line — normalize, don't multiply

The activation's norm (~130) and a typical token embedding's norm are in the same ballpark, but not identical. The verbalizer was trained on activations whose norm was **set to 150**. So we rescale the vector's length to 150 and keep its direction:

```
normalize_to(v) =  v * (150 / ||v||)      # length becomes exactly 150
```
The classic bug is to write `v * 150` instead — that makes the norm ~19,500, **130× too big**, a vector from a galaxy the model has never seen. The next cell runs both versions once and places their outputs side by side.

In [6]:
from html import escape
from IPython.display import HTML, display

results = {}
for BUG in (False, True):
    scale_fn = (lambda v: v * INJECT_SCALE) if BUG else normalize_to
    injected = scale_fn(activation.to(emb.dtype))
    emb2 = emb.clone()
    emb2[0, pos, :] = injected
    with torch.no_grad():
        out = model.generate(input_ids=input_ids, inputs_embeds=emb2,
                             attention_mask=torch.ones_like(input_ids),
                             max_new_tokens=120, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    seq = out[0]; gen = seq[len(ids):] if seq.shape[0] > len(ids) else seq
    text = tok.decode(gen, skip_special_tokens=True).split("</explanation>")[0].strip()
    results[BUG] = (injected.float().norm().item(), text)

def panel(title, result, color):
    norm, text = result
    return (f'<div style="border:2px solid {color};padding:12px;border-radius:8px">'
            f'<b>{title}</b><br>injected norm: {norm:.1f}'
            f'<pre style="white-space:pre-wrap">{escape(text)}</pre></div>')

display(HTML('<div style="display:grid;grid-template-columns:1fr 1fr;gap:14px">'
             + panel('CORRECT: normalize TO 150', results[False], '#2e8b57')
             + panel('BUG: multiply BY 150', results[True], '#b22222')
             + '</div>'))

The right panel's norm jumps ~100×, yet its readout can stay **fluent** while detaching from the vector: confident bullets about another topic entirely (measured on GPU 2026-07-10, this notebook configuration, 4-bit). No error, no gibberish, no warning. That single confusion (`normalize TO` vs `multiply BY`) is mistake #1 in the history-of-pain table: a readout that lies fluently is worse than one that crashes.

## The second mistake — the hook that eats itself

`read_activation` guards its hook with `if "h" not in grab`. Here's why. During `generate()` the forward hook fires on **every** generated token, so without the guard `grab["h"]` ends up holding the *last generated token's* activation instead of the prompt state we intended to measure. Watch the guard matter:

In [7]:
def read_unguarded(prompt, layer=LAYER):
    chat = tok.apply_chat_template([{"role":"user","content":prompt}],
                                   tokenize=False, add_generation_prompt=True)
    inp = tok(chat, return_tensors="pt").to(device)
    grab = {}
    def hook(m, i, o):
        h = o[0] if isinstance(o, tuple) else o
        grab["h"] = h[:, -1, :].detach()   # NO guard: overwritten every step
    handle = get_layers(model)[layer].register_forward_hook(hook)
    try:
        with model.disable_adapter(), torch.no_grad():
            model.generate(**inp, max_new_tokens=40, do_sample=False,
                           pad_token_id=tok.eos_token_id)
    finally:
        handle.remove()
    return grab["h"].squeeze(0)

p = "Explain how a hash map handles collisions."
good = read_activation(p)[0]
bad  = read_unguarded(p)
print("guarded (prompt state)   ->", describe(good))
print()
print("unguarded (last gen tok) ->", describe(bad))

guarded (prompt state)   -> - Hash table: data structure for key-value storage
- "how does hash table works" frame: mechanism-explanation response
- "in terms of hashing algorithm" constraint: technical precision (hashing function, collision resolution)
- "in computer science" context: formal, neutral register
- Response strategy: step-by-step procedural explanation (hashing, storing, retrieving keys and values) with a focus on collision handling (chaining or probing)



unguarded (last gen tok) -> - C++ code generation: "Write a function" with `#include`, `class HashTable`, and `int hash(string s)` as structural anchors
- Key-value storage schema: "hashes a string to an integer index" with `std::vector<std::string>` as output type
- Response strategy: implementation of `hash` function with modulo operation on ASCII values
- Tension between "hash function" (general cryptographic) and "hashes a string to an integer index" (simple arithmetic modulo)
- Output format: "return the index" constraining return type to


## The depth number is an input, not a label

`DEPTH_PCT = 71` is fed *into* the verbalizer's prompt. The current universal Qwen AV was trained across all 28 layers using thirteen rounded depth tags; **71%** is the tag paired with layer 20. Re-labeling the same layer-20 vector as 40% or 96% creates a mismatched vector/tag pair. Both tags occurred in training, but not with this vector, so the changed caption demonstrates that depth is part of the interface. It does not mean another tag is intrinsically better or worse.

In [8]:
act, _ = read_activation("Explain how a hash map handles collisions.")
for d in [71, 40, 96, 10]:
    print(f"depth told = {d:>2}%  ->  {describe(act, depth=d)}")

depth told = 71%  ->  - Hash table: data structure for key-value storage
- "how does hash table works" frame: mechanism-explanation response
- "in terms of hashing algorithm" constraint: technical precision (hashing function, collision resolution)
- "in computer science" context: formal, neutral register
- Response strategy: step-by-step procedural explanation (hashing, storing, retrieving keys and values) with a focus on collision handling (chaining or probing)


depth told = 40%  ->  -
- "How does X work?" frame: technical-explanatory response structure
- "hashing algorithm" and "nodes" co-indexed: distributed systems and data structures (hash tables)
- "in a hash table based algorithm" narrowing: node-level behavior during insertion or lookup
- "how nodes are organized" vs. "how hashing works": competing response strategies (node organization vs. hashing mechanism)
- "in detail" reinforcing: thorough, step-by-step explanation
- "hash code" as a noun phrase: clarifying the subject of the question
- "hash code


depth told = 96%  ->  -
- Technical CS question: "How does a hash table work in a hash map" with "hash table" and "hash map" as key concepts
- Response structure: "In computer science" or "A hash table works" as opening tokens, followed by "collision resolution" as a sub-question
- Topic classification: "hash table" and "hash map" as data-structure and algorithmic concepts, with "work" and "how does it work" indicating procedural, step-by-step explanation
- Tension between "hash table" and "hash map": "hash table


depth told = 10%  ->  -
- Technical query frame: "How does X work?" with "hashing algorithm" and "nodes" as key terms
- Contextual co-occurrence: "hashing algorithm" and "nodes" in distributed systems and data structures
- Response strategy: explanatory, step-by-step answer about hash functions and node distribution
- Competing interpretations: "how does hashing algorithm works" (algorithmic vs. operational) and "in distributed systems" (contextual constraint)
- Next-token predictions: technical vocabulary ("hash function", "bucket", "consistent hashing") and procedural framing ("works


---
### ✅ Self-check
Expected: `BUG=False` gives an on-topic caption and injected norm ≈ 150; `BUG=True` gives norm ≈ 19,500 and fluent but unrelated output; the **guarded** and **unguarded** hooks capture different states; changing the depth tag changes the caption even though the vector is identical.

In [9]:
v, _ = read_activation("Explain how a hash map handles collisions.")
assert abs(normalize_to(v).float().norm().item() - 150) < 1.0
assert (v.float().norm() * 150 > 1000)   # the multiply-bug really is huge
print("self-check: normalize_to lands on 150, multiply-by-150 does not ✓")

self-check: normalize_to lands on 150, multiply-by-150 does not ✓
